# 

# 03_compare_SVM_to_BERT

In this notebook, we will train a SVM model on the reduced dataset used the BERT performances for our task. We will then be able to compare the SVM performances with BERT for our task. 


**Inputs**

The reduced dataset in csv format, constructed in the first notebook. 

**Outputs**

The SVM model and its evaluation. 

## 1. Library import 

In [ ]:
import pandas as pd 
import joblib
import random 

from sklearn.preprocessing import StandardScaler
from sklearn import metrics

from scipy.sparse import hstack, vstack
from scipy.sparse import csr_matrix
from sklearn import svm

from sklearn.model_selection import permutation_test_score
from sklearn.metrics import accuracy_score
import numpy as np

## 2. Data import  

In [ ]:
# Download train data 
df_gpt = pd.read_csv("metrics//data//df_gpt.csv")
display(df_gpt)
# Download test data 
df_test = pd.read_csv("metrics//data//df_test_gpt.csv")

,text,paragraph,categorie,nombre_mots,tokens,pos_tags,function_word_freq,type_token_ratio,avg_word_length,noun_ratio,verb_ratio,adj_ratio,adv_ratio,exclamation_freq,question_freq,comma_freq,avg_sentence_length,mean_tfidf,max_tfidf,rare_word_count
0,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",Here and there adult wizards and witches were ...,1,124,"['Here', 'and', 'there', 'adult', 'wizards', '...","['ADV', 'CCONJ', 'ADV', 'NOUN', 'NOUN', 'CCONJ...",0.422535,0.690141,4.584507,0.211268,0.112676,0.119718,0.021127,0.0,0.000000,0.049296,35.500000,0.005227,0.326540,2
1,Harry Potter and the Half-Blood Prince - Joann...,Harry let out a hastily stifled gasp. Voldemor...,1,118,"['Harry', 'let', 'out', 'a', 'hastily', 'stifl...","['PROPN', 'VERB', 'ADP', 'DET', 'ADV', 'VERB',...",0.511278,0.586466,3.917293,0.150376,0.097744,0.067669,0.097744,0.0,0.000000,0.045113,26.600000,0.004968,0.288962,0
2,Amor Vincit Omnia - Twin_Flame_Blues.txt,"“Listen to me, Hermione. You have not failed. ...",0,115,"['“', 'Listen', 'to', 'me', ',', 'Hermione', '...","['PUNCT', 'VERB', 'ADP', 'PRON', 'PUNCT', 'PRO...",0.587838,0.560811,3.317568,0.074324,0.121622,0.033784,0.081081,0.0,0.013514,0.067568,13.454545,0.004474,0.503270,1
3,The Cadence of Part-time Poets - motswolo.txt,"And so, as the November chill gave way to the ...",0,100,"['And', 'so', ',', 'as', 'the', 'November', 'c...","['CCONJ', 'ADV', 'PUNCT', 'SCONJ', 'DET', 'PRO...",0.543103,0.681034,3.913793,0.137931,0.094828,0.077586,0.051724,0.0,0.000000,0.068966,29.000000,0.004332,0.431182,1
4,Turn - Saras_Girl.txt,"“I’d forgotten about that,” Harry sighs, hangi...",0,117,"['“', 'I', '’d', 'forgotten', 'about', 'that',...","['PUNCT', 'PRON', 'AUX', 'VERB', 'ADP', 'PRON'...",0.523490,0.671141,3.899329,0.087248,0.134228,0.053691,0.067114,0.0,0.026846,0.080537,21.285714,0.004683,0.277918,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,The Debt of Time - ShayaLonnie.txt,"She and Sirius stayed up all night, keeping wa...",0,111,"['She', 'and', 'Sirius', 'stayed', 'up', 'all'...","['PRON', 'CCONJ', 'PROPN', 'VERB', 'ADP', 'DET...",0.495935,0.650407,3.934959,0.130081,0.154472,0.016260,0.024390,0.0,0.000000,0.040650,24.600000,0.004729,0.402723,2
396,Secrets and Masks - Emerald_Slytherin.txt,"""Oh, well then that'll mean Hermione will be o...",0,101,"['""', 'Oh', ',', 'well', 'then', 'that', ""'ll""...","['PUNCT', 'INTJ', 'PUNCT', 'INTJ', 'ADV', 'PRO...",0.536585,0.666667,3.455285,0.121951,0.138211,0.032520,0.032520,0.0,0.008130,0.040650,20.500000,0.004940,0.319645,1
397,Lily's Boy - SomewheresSword.txt,"“Like I said, I could be wrong, in which case,...",0,118,"['“', 'Like', 'I', 'said', ',', 'I', 'could', ...","['PUNCT', 'INTJ', 'PRON', 'VERB', 'PUNCT', 'PR...",0.573333,0.606667,3.193333,0.093333,0.140000,0.046667,0.033333,0.0,0.000000,0.073333,21.428571,0.005576,0.337152,1
398,Harry Potter and The Order of the Phoenix - Jo...,He felt as though the memory of it was eating ...,1,145,"['He', 'felt', 'as', 'though', 'the', 'memory'...","['PRON', 'VERB', 'SCONJ', 'SCONJ', 'DET', 'NOU...",0.493902,0.591463,4.048780,0.109756,0.091463,0.036585,0.054878,0.0,0.012195,0.042683,20.500000,0.005636,0.397818,1


In [ ]:
# Load tf-idf vectorizer
tfidf = joblib.load("metrics/data/tfidf_vectorizer.pkl")

## 3. Split train test 

### 3.1 Training columns 

In [ ]:

stylometric_columns = [
    "function_word_freq",
    "type_token_ratio",
    "avg_word_length",
    "noun_ratio",
    "verb_ratio",
    "adj_ratio",
    "adv_ratio",
    "exclamation_freq",
    "question_freq",
    "comma_freq",
    "avg_sentence_length"
]



### 3.2 X_train and y_train 

In [ ]:
# Construct the tf-idf matrix
X_tfidf_gpt = tfidf.transform(df_gpt["paragraph"])
X_tfidf = tfidf.transform(df_test["paragraph"])

In [ ]:

# We first create X_train and y_train by using df_gpt
X_stylo = df_gpt[stylometric_columns].values


X_train = hstack([
    X_tfidf_gpt,
    csr_matrix(X_stylo)
])

y_train = df_gpt["categorie"]


### 3.3 X_test and y_test 

In [ ]:

# Stylometric features
X_stylo = df_test[stylometric_columns].values

# TF-IDF features
X_tfidf = tfidf.transform(df_test["paragraph"])

# Combined feature matrix
X_test = hstack([
    X_tfidf,
    csr_matrix(X_stylo)
])

# Labels
y_test = df_test["categorie"]


### 3.4 Normalize data 

In [ ]:
# We normalize the data 
scaler = StandardScaler(with_mean=False)  

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4. Data modeling 

### 4.1 Naive model 

In [ ]:
# We compute a naive model by predicting a random categorie each time 
y_pred_naive = [random.choice([0, 1])]*len(y_test)

print("---Performances of the Naive Model---\n")
print("Accuracy:" ,metrics.accuracy_score(y_pred_naive, y_test))

---Performances of the Naive Model---

Accuracy: 0.5


### 4.2 SMV model 

### 4.2.1 Training 

In [ ]:

#Create a svm Classifier
model_svm = svm.SVC(kernel='poly', class_weight="balanced") # Polynomial Kernel

#Train the model using the training sets
model_svm.fit(X_train, y_train)

#Predict the response for test dataset
y_pred_svm_train = model_svm.predict(X_train)
y_pred_svm = model_svm.predict(X_test)

### 4.2.2 Evaluation 

In [ ]:
# Print the evaluated performances 

print("---Performances of the model on train data---\n")
print("Accuracy:",metrics.accuracy_score(y_pred_svm_train, y_train))
print("F1-Score:",metrics.f1_score(y_pred_svm_train, y_train))
print("Recall",metrics.recall_score(y_pred_svm_train, y_train))

print("\n\n---Performances of the model on test data---\n")

print("Accuracy:",metrics.accuracy_score(y_pred_svm, y_test))
print("F1-Score:",metrics.f1_score(y_pred_svm, y_test))
print("Recall",metrics.recall_score(y_pred_svm, y_test))


---Performances of the model on train data---

Accuracy: 1.0
F1-Score: 1.0
Recall 1.0


---Performances of the model on test data---

Accuracy: 0.7783333333333333
F1-Score: 0.8017883755588674
Recall 0.7250673854447439


### 4.2.3 p-value 

In [ ]:


# Combine train + test (IMPORTANT pour le test de permutation)
X = vstack([X_train, X_test])
y = np.concatenate([y_train, y_test])

# Define the model again (same configuration)
model = svm.SVC(kernel='poly', class_weight="balanced")

# Compute permutation test
score, permutation_scores, p_value = permutation_test_score(
    model,
    X,
    y,
    scoring="accuracy",
    cv=5,              # cross-validation
    n_permutations=100,  # increase to 1000 for final report
    n_jobs=-1
)

print("Model accuracy:", score)
print("Permutation p-value:", p_value)

Model accuracy: 0.7939999999999999
Permutation p-value: 0.009900990099009901
